Первая модель тест

In [ ]:
!pip install transformers torch tqdm nbformat

In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:

!pip install -q accelerate
!huggingface-cli download facebook/m2m100_418M --local-dir ./m2m100_model --local-dir-use-symlinks False

from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
import nbformat
from pathlib import Path
from tqdm import tqdm
import torch

model_name = "facebook/m2m100_418M"
model = M2M100ForConditionalGeneration.from_pretrained("./m2m100_model")
tokenizer = M2M100Tokenizer.from_pretrained("./m2m100_model")

device = "cuda"
model = model.to(device)

tokenizer.src_lang = "ru"
tgt_lang = "en"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/commands/download.py:141: FutureWarning: Ignoring --local-dir-use-symlinks. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Fetching 10 files:   0% 0/10 [00:00<?, ?it/s]Downloading 'README.md' to 'm2m100_model/.cache/huggingface/download/Xn7B-BWUGOee2Y6hCZtEhtFu4BE=.98b99edb1de68441bb0c87a9645c8f2e35ea34d0.incomplete'

README.md: 4.60kB [00:00, 15.4MB/s]

.gitattributes: 100% 690/690 [00:00<00:00, 5.28MB/s]Downloading 'pytorch_model.bin' to 'm2m100_model/.cache/huggingface/download/Q1p2l2BzM1m6P5jKvr8WTq1TUio=.d907ea45e4e4b9db163382a6674f6218b3c59566fe06d77f4055c208b4e87ed1.incomplete'

Download complete. Moving file to m2m100_model/.gitattributes
Fetching 10 files:  10% 1/10 [00:00<00:03,  2.93it/s]
generation_config.json: 100% 233/233 [00:00<00:00, 1.87MB/s]
Download complete. Moving file to m2m100_model/README.md



In [ ]:
def translate_text(text: str) -> str:
    if not text.strip():
        return text
    encoded = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    generated_tokens = model.generate(
        **encoded,
        forced_bos_token_id=tokenizer.get_lang_id(tgt_lang),
        max_length=512
    )
    return tokenizer.decode(generated_tokens[0], skip_special_tokens=True)


import re
import nbformat

def translate_notebook(input_path: Path, output_path: Path):
    nb = nbformat.read(input_path, as_version=4)

    for cell in nb["cells"]:
        if cell["cell_type"] == "markdown":
            cell["source"] = translate_text(cell["source"])

        elif cell["cell_type"] == "code":
            lines = cell["source"].split("\n")
            new_lines = []
            for line in lines:
                if "#" in line:
                    parts = line.split("#", 1)
                    before_comment = parts[0]
                    comment = parts[1]
                    comment_translated = translate_text(comment)
                    line = before_comment + "# " + comment_translated

                matches = re.findall(r'(["\'])(.*?)(\1)', line)
                for full, inner, _ in matches:
                    if re.search(r'[А-Яа-яЁё]', inner):
                        translated = translate_text(inner)
                        line = line.replace(inner, translated)

                new_lines.append(line)
            cell["source"] = "\n".join(new_lines)

    nbformat.write(nb, output_path)

In [ ]:
import requests
import random
from pathlib import Path

user = "alyasaff13-cmyk"
repo = "database_notebook"

input_dir = Path("/content/notebooks")
input_dir.mkdir(exist_ok=True)

# Получаем список всех файлов репозитория через GitHub API
url = f"https://api.github.com/repos/{user}/{repo}/git/trees/main?recursive=1"
files = requests.get(url).json()["tree"]

nb_files = [f["path"] for f in files if f["path"].endswith(".ipynb")]
print(f"Всего {len(nb_files)} ноутбуков в репозитории")

sample_files = random.sample(nb_files, min(10, len(nb_files)))

for path in sample_files:
    raw_url = f"https://raw.githubusercontent.com/{user}/{repo}/main/{path}"
    name = Path(path).name
    r = requests.get(raw_url)
    with open(input_dir / name, "wb") as f:
        f.write(r.content)
print(input_dir)

Всего 131 ноутбуков в репозитории
/content/notebooks


In [ ]:
output_dir = Path("/content/translated_notebooks")
output_dir.mkdir(exist_ok=True)

for notebook_path in tqdm(list(input_dir.glob("*.ipynb"))):
    output_path = output_dir / notebook_path.name
    translate_notebook(notebook_path, output_path)

print(output_dir)

100%|██████████| 10/10 [04:25<00:00, 26.57s/it]

/content/translated_notebooks


In [ ]:
from google.colab import files
import shutil

shutil.make_archive("/content/translated_notebooks", "zip", output_dir)
files.download("/content/translated_notebooks.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
shutil.make_archive("/content/notebooks", "zip", input_dir)
files.download("/content/notebooks")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



---



In [1]:
!pip install transformers torch tqdm nbformat accelerate sentencepiece


In [2]:
import torch
print("GPU available:", torch.cuda.is_available())

!huggingface-cli download facebook/nllb-200-distilled-600M \
    --local-dir ./nllb_model \
    --local-dir-use-symlinks False


GPU available: True
/usr/local/lib/python3.12/dist-packages/huggingface_hub/commands/download.py:141: FutureWarning: Ignoring --local-dir-use-symlinks. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Fetching 9 files:   0% 0/9 [00:00<?, ?it/s]Downloading 'generation_config.json' to 'nllb_model/.cache/huggingface/download/3EVKVggOldJcKSsGjSdoUCN1AyQ=.4c6d168201607a22255bee55a6ed092b0e3ed7a1.incomplete'

README.md: 7.67kB [00:00, 21.4MB/s]


generation_config.json: 100% 189/189 [00:00<00:00, 1.35MB/s]

config.json: 100% 846/846 [00:00<00:00, 5.72MB/s]
Download complete. Moving file to nllb_model/README.md
Download complete. Moving file to nllb_model/generation_config.json
Download complete. Moving file to nllb_model/config.json

.gitattributes: 1.28kB [00:00, 8.86MB/s]
Download complete. Moving file to nllb_model/.gitattributes
Fetching 9 files:  11% 1/9 [00:00<00:01,  4.56i

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_dir = "./nllb_model"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

src_lang = "rus_Cyrl"
tgt_lang = "eng_Latn"


In [4]:
def translate_text(text: str) -> str:
    if not text.strip():
        return text

    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    generated = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
        max_length=1024
    )

    return tokenizer.decode(generated[0], skip_special_tokens=True)


In [5]:
import re
import nbformat
from pathlib import Path

def translate_notebook(input_path: Path, output_path: Path):
    nb = nbformat.read(input_path, as_version=4)

    for cell in nb["cells"]:

        # Markdown переводим целиком
        if cell["cell_type"] == "markdown":
            cell["source"] = translate_text(cell["source"])

        # Переводим комментарии и строки в коде
        elif cell["cell_type"] == "code":
            lines = cell["source"].split("\n")
            new_lines = []

            for line in lines:

                # Перевод комментариев после #
                if "#" in line:
                    before, comment = line.split("#", 1)
                    comment_tr = translate_text(comment)
                    line = before + "# " + comment_tr

                # Перевод строковых литералов
                matches = re.findall(r'(["\'])(.*?)(\1)', line)
                for full, inner, _ in matches:
                    if re.search("[А-Яа-яЁё]", inner):
                        line = line.replace(inner, translate_text(inner))

                new_lines.append(line)

            cell["source"] = "\n".join(new_lines)

    nbformat.write(nb, output_path)


In [6]:
import requests
import random
from pathlib import Path

user = "alyasaff13-cmyk"
repo = "database_notebook"

input_dir = Path("/content/notebooks")
input_dir.mkdir(exist_ok=True)

url = f"https://api.github.com/repos/{user}/{repo}/git/trees/main?recursive=1"
files = requests.get(url).json()["tree"]

nb_files = [f["path"] for f in files if f["path"].endswith(".ipynb")]
print(f"Всего ноутбуков: {len(nb_files)}")

sample_files = random.sample(nb_files, min(10, len(nb_files)))

for path in sample_files:
    raw_url = f"https://raw.githubusercontent.com/{user}/{repo}/main/{path}"
    name = Path(path).name
    data = requests.get(raw_url).content
    with open(input_dir / name, "wb") as f:
        f.write(data)

print("Скачано в:", input_dir)


Всего ноутбуков: 151
Скачано в: /content/notebooks


In [7]:
from tqdm import tqdm

output_dir = Path("/content/translated_notebooks")
output_dir.mkdir(exist_ok=True)

for nb_path in tqdm(list(input_dir.glob("*.ipynb")), desc="Перевод"):
    out = output_dir / nb_path.name
    translate_notebook(nb_path, out)

print("Готово:", output_dir)


Перевод: 100%|██████████| 10/10 [04:38<00:00, 27.83s/it]

Готово: /content/translated_notebooks


In [8]:
from google.colab import files
import shutil

shutil.make_archive("/content/translated_notebooks", "zip", output_dir)
files.download("/content/translated_notebooks.zip")

shutil.make_archive("/content/notebooks", "zip", input_dir)
files.download("/content/notebooks.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>